 # Bronze→Silver Transformation

In [6]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

BASE = r"C:\Users\PC 12\Motor-Insurance-Quote-to-Policy-Conversion\data"
np.random.seed(42)

# ==========================================================
# Load + align the two source populations
# (from last turn — establishes one shared entity chain)
# ==========================================================

raw_vehicle_df = pd.read_parquet(f"{BASE}\\bronze\\vehicle_insurance\\vehicle.parquet")
claims_df = pd.read_parquet(f"{BASE}\\bronze\\analytics_vidhya_claims\\combined_claims.parquet")

claims_df = claims_df.sort_values("age_of_policyholder").reset_index(drop=True)
n = len(claims_df)

vehicle_sorted = raw_vehicle_df.sort_values("customer_age").reset_index(drop=True)
sample_idx = np.linspace(0, len(vehicle_sorted) - 1, n).round().astype(int)
crosssell_df = vehicle_sorted.iloc[sample_idx].reset_index(drop=True)

assert len(crosssell_df) == len(claims_df) == n
print(f"Aligned population size: {n:,}")

# ------------------------------------------------------------
# is_claim check — resolves the open Bronze question directly
# ------------------------------------------------------------
n_missing_claim = claims_df["is_claim"].isna().sum()
print(f"is_claim missing: {n_missing_claim:,} of {n:,}")
if n_missing_claim > 0:
    print("WARNING: filling missing is_claim as 0 (no claim) — verify this is the "
          "correct treatment before trusting claim_flag downstream.")
claims_df["is_claim"] = claims_df["is_claim"].fillna(0)

# ==========================================================
# Customer Silver — from crosssell_df (aligned)
# ==========================================================

customer_df = crosssell_df[
    ["gender", "customer_age", "has_driving_license", "previously_insured", "region"]
].copy()

customer_df.insert(0, "customer_sk", range(1, n + 1))

# right=False so age 18 lands in the 18-25 band instead of NaN
customer_df["age_band"] = pd.cut(
    customer_df["customer_age"],
    bins=[18, 26, 36, 46, 61, 200],
    right=False,
    labels=["18-25", "26-35", "36-45", "46-60", "60+"]
)

os.makedirs(f"{BASE}\\silver", exist_ok=True)
customer_df.to_csv(f"{BASE}\\silver\\customer.csv", index=False)
print("customer.csv created —", customer_df.shape)

# ==========================================================
# Vehicle Silver — from claims_df (aligned)
# Renamed variable to avoid the vehicle_df collision from before
# ==========================================================

claims_vehicle_df = claims_df.rename(columns={
    "age_of_car": "vehicle_age",
    "transmission_type": "transmission",
    "steering_type": "steering"
})[[
    "policy_id", "vehicle_age", "make", "model", "fuel_type", "segment",
    "engine_type", "displacement", "cylinder", "transmission", "steering",
    "length", "width", "height", "gross_weight"
]].copy()

claims_vehicle_df.insert(0, "vehicle_sk", range(1, n + 1))
# policy_id kept intentionally — real join key within the claims-derived
# tables (vehicle/safety/policy/claim all come from the same source rows)

claims_vehicle_df.to_csv(f"{BASE}\\silver\\vehicle.csv", index=False)
print("vehicle.csv created —", claims_vehicle_df.shape)

# ==========================================================
# Safety Table — rescaled to 0-100 so Gold's Excellent/Good/
# Average/Poor thresholds (90/75/60) are actually reachable
# ==========================================================

safety_df = claims_df[[
    "policy_id", "airbags", "is_esc", "is_tpms", "is_parking_sensors",
    "is_parking_camera", "is_brake_assist", "is_power_steering",
    "is_speed_alert", "ncap_rating"
]].copy()

binary_columns = [
    "is_esc", "is_tpms", "is_parking_sensors", "is_parking_camera",
    "is_brake_assist", "is_power_steering", "is_speed_alert"
]
for col in binary_columns:
    safety_df[col] = safety_df[col].map({"Yes": 1, "No": 0, True: 1, False: 0})

raw_score = (
    safety_df["airbags"] / 6 * 30          # airbags: 0-6 → up to 30 pts
    + safety_df[binary_columns].sum(axis=1) / 7 * 40   # 7 binary features → up to 40 pts
    + safety_df["ncap_rating"] / 5 * 30    # ncap: 0-5 → up to 30 pts
)
safety_df["safety_score"] = raw_score.round(1)  # now genuinely spans ~0-100

safety_df.to_csv(f"{BASE}\\silver\\vehicle_safety.csv", index=False)
print("vehicle_safety.csv created —", safety_df.shape)

# ==========================================================
# Channel Lookup — mapping now applied BEFORE save
# (previous version saved placeholder names, mapped names
# only ever existed in memory)
# ==========================================================

channels = sorted(crosssell_df["sales_channel"].dropna().unique())

channel_mapping = {
    "Agent": range(1, 21), "Website": range(21, 41), "Mobile App": range(41, 61),
    "Branch": range(61, 81), "Call Center": range(81, 101), "Aggregator": range(101, 121),
    "Partner": range(121, 141), "Corporate": range(141, 1000)
}

def map_channel(channel_id):
    if pd.isna(channel_id):
        return "Unknown"
    channel_id = int(channel_id)
    for name, values in channel_mapping.items():
        if channel_id in values:
            return name
    return "Other"

lookup = pd.DataFrame()
lookup["channel_id"] = channels
lookup["channel_name"] = [map_channel(c) for c in channels]  # mapped before save now

lookup.to_csv(f"{BASE}\\silver\\channel_lookup.csv", index=False)
print("channel_lookup.csv created —", lookup.shape)
print("Note: this channel taxonomy is a fabricated demo convention, not real "
      "insurer business logic — worth one line in the data dictionary.")

# ==========================================================
# Quote Silver — now carries a real vehicle_sk, and both
# accepted_offer and conversion_flag exist explicitly
# ==========================================================

quote_df = pd.DataFrame()
quote_df["quote_sk"] = range(1, n + 1)
quote_df["quote_id"] = [f"Q{100000+i}" for i in range(n)]
quote_df["customer_sk"] = range(1, n + 1)
quote_df["vehicle_sk"] = range(1, n + 1)  # legitimate now — same aligned population

vehicle_age_num = crosssell_df["vehicle_age"].map({
    "< 1 Year": 0.5, "1-2 Year": 1.5, "> 2 Years": 3.0
}).fillna(1.0)

base_premium = 3500 + crosssell_df["customer_age"] * 20 + vehicle_age_num * 500
quote_df["quoted_premium"] = (base_premium + np.random.randint(-500, 500, n)).round(0)

quote_df["sales_channel"] = crosssell_df["sales_channel"]
quote_df["days_since_last_contact"] = crosssell_df["days_since_last_contact"]

# accepted_offer: the raw Cross-Sell signal (customer said yes to the quote)
# accepted_offer: raw Cross-Sell interest signal (customer said yes to the quote)
quote_df["accepted_offer"] = crosssell_df["accepted_offer"]

# Interest and issuance are NOT the same event. Simulate a second
# attrition layer for customers who accepted the offer but dropped
# before a policy was actually issued — document upload, underwriting
# decline, or payment failure. This is the funnel the case is actually
# asking you to analyze; without it there's no gap between accepted_offer
# and conversion_flag to explain.
np.random.seed(43)
post_accept_outcome = np.random.choice(
    ["Converted", "Document Upload", "Underwriting", "Payment"],
    n,
    p=[0.80, 0.08, 0.07, 0.05]  # 20% of accepters still drop off after saying yes
)

quote_df["conversion_flag"] = np.where(
    (quote_df["accepted_offer"] == 1) & (post_accept_outcome == "Converted"), 1, 0
)

quote_df["quote_status"] = np.where(quote_df["conversion_flag"] == 1, "Converted", "Abandoned")

# quote_stage now shows WHERE each group actually sits, not a single
# uniform random draw for everyone who didn't convert
quote_df["quote_stage"] = np.select(
    [
        quote_df["conversion_flag"] == 1,
        (quote_df["accepted_offer"] == 1) & (post_accept_outcome != "Converted"),
        quote_df["accepted_offer"] == 0,
    ],
    [
        "Policy Issued",
        post_accept_outcome,  # Document Upload / Underwriting / Payment — the real drop stage
        np.random.choice(["Premium Calculation", "Document Upload"], n, p=[0.6, 0.4]),
    ],
        default="Unknown"   # <- this line fixes it
)

print("accepted_offer rate:", quote_df["accepted_offer"].mean().round(3))
print("conversion_flag rate:", quote_df["conversion_flag"].mean().round(3))
print(quote_df.loc[quote_df["accepted_offer"] == 1, "quote_stage"].value_counts())
quote_df["device_type"] = np.random.choice(["Mobile", "Desktop", "Tablet"], n, p=[0.6, 0.3, 0.1])
quote_df["quote_source"] = np.random.choice(["Website", "Aggregator", "Agent", "Mobile App"], n)

# NOTE: unresolved — confirm dim_date's actual range before trusting this join.
# Two turns ago the executive_dashboard output showed date_sk starting at
# 2022-01-01; this generates all 2025. If dim_date doesn't cover 2025, every
# quote here will fail to join. Flagging, not guessing — revisit at the date-dim notebook.
start_date = datetime(2025, 1, 1)
quote_df["quote_date"] = [start_date + timedelta(days=int(np.random.randint(0, 365))) for _ in range(n)]

quote_df["load_date"] = pd.Timestamp.now()
quote_df["source_system"] = "Vehicle Insurance"

quote_df.to_csv(f"{BASE}\\silver\\quote.csv", index=False)
print("quote.csv created —", quote_df.shape)

# ==========================================================
# Policy Silver — now carries customer_sk (valid, post-alignment)
# ==========================================================

policy_df = pd.DataFrame()
policy_df["policy_sk"] = range(1, n + 1)
policy_df["policy_id"] = claims_df["policy_id"]
policy_df["customer_sk"] = range(1, n + 1)  # now legitimate
policy_df["policy_tenure"] = claims_df["policy_tenure"]
policy_df["policy_tenure_band"] = pd.cut(
    policy_df["policy_tenure"], bins=[0, 0.5, 1, 5],
    labels=["< 6 Months", "6-12 Months", "> 12 Months"]
)
# tenure_category intentionally not duplicated here — this was the same
# field twice in the original Gold dim_policy; keep only policy_tenure_band

policy_df["policy_type"] = np.random.choice(["Third Party", "Comprehensive"], n, p=[0.35, 0.65])
policy_df["policy_status"] = np.random.choice(["Active", "Expired", "Cancelled"], n, p=[0.90, 0.08, 0.02])
policy_df["coverage_type"] = np.random.choice(["Basic", "Silver", "Gold"], n)
policy_df["load_date"] = pd.Timestamp.now()
policy_df["source_system"] = "Insurance Claims"

policy_df.to_csv(f"{BASE}\\silver\\policy.csv", index=False)
print("policy.csv created —", policy_df.shape)

# ==========================================================
# Claim Silver — customer_sk + vehicle_sk now valid, fraud_risk
# decorrelated from claim_amount so it isn't just claim_severity
# restated under a different name
# ==========================================================

claim_df = pd.DataFrame()
claim_df["claim_sk"] = range(1, n + 1)
claim_df["claim_id"] = [f"CLM{100000+i}" for i in range(n)]
claim_df["policy_id"] = claims_df["policy_id"]
claim_df["customer_sk"] = range(1, n + 1)
claim_df["vehicle_sk"] = range(1, n + 1)

claim_df["claim_flag"] = claims_df["is_claim"]

claim_df["claim_amount"] = np.where(
    claim_df["claim_flag"] == 1, np.random.randint(5000, 150000, n), 0
)
claim_df["settlement_days"] = np.where(
    claim_df["claim_flag"] == 1, np.random.randint(2, 30, n), 0
)
claim_df["claim_severity"] = pd.cut(
    claim_df["claim_amount"], bins=[-1, 10000, 50000, 200000], labels=["Low", "Medium", "High"]
)

# fraud_risk: now a function of settlement speed + amount together,
# with noise, instead of a pure restatement of claim_severity
fraud_score = (
    (claim_df["claim_amount"] > 100000).astype(int)
    + (claim_df["settlement_days"] < 5).astype(int)  # suspiciously fast settlement
    + (np.random.random(n) < 0.05).astype(int)        # random noise component
)
claim_df["fraud_risk"] = np.where(claim_df["claim_flag"] == 1,
                                   np.where(fraud_score >= 2, "High", "Low"), "Low")

claim_df["load_date"] = pd.Timestamp.now()
claim_df["source_system"] = "Insurance Claims"

claim_df.to_csv(f"{BASE}\\silver\\claim.csv", index=False)
print("claim.csv created —", claim_df.shape)

print("\nAll Silver tables built on one aligned entity chain, n =", n)

Aligned population size: 97,655
is_claim missing: 39,063 of 97,655
customer.csv created — (97655, 7)
vehicle.csv created — (97655, 16)
vehicle_safety.csv created — (97655, 11)
channel_lookup.csv created — (139, 2)
Note: this channel taxonomy is a fabricated demo convention, not real insurer business logic — worth one line in the data dictionary.
accepted_offer rate: 0.123
conversion_flag rate: 0.098
quote_stage
Policy Issued      9596
Document Upload     948
Underwriting        860
Payment             637
Name: count, dtype: int64
quote.csv created — (97655, 16)
policy.csv created — (97655, 10)
claim.csv created — (97655, 12)

All Silver tables built on one aligned entity chain, n = 97655
